# 15 — Fraud Tuning & Calibration

## Objectif

L'étape précédente a montré que XGBoost et Random Forest présentent
les meilleures performances pour la détection de fraude.

Cette étape vise à :

1. ajuster les hyperparamètres uniquement à partir des données 1994 ;
2. comparer XGBoost et Random Forest ;
3. étudier l'effet de la pondération des classes ;
4. calibrer les probabilités ;
5. évaluer les configurations sur l'année 1995 ;
6. préparer la sélection du candidat final.

L'année 1996 reste totalement exclue du processus de sélection.

In [1]:
import pandas as pd
import numpy as np

import json
import time

from pathlib import Path

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier


RANDOM_STATE = 42

In [2]:
cwd = Path.cwd()

possible_processed_paths = [
    cwd / "data" / "processed",
    cwd / "data-science" / "data" / "processed",
    cwd.parent / "data" / "processed"
]

PROCESSED_DIR = None

for path in possible_processed_paths:
    if path.exists():
        PROCESSED_DIR = path
        break

if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "Impossible de localiser data/processed"
    )


DATA_DIR = PROCESSED_DIR.parent
METADATA_DIR = DATA_DIR / "metadata"

REPORT_DIR = (
    DATA_DIR.parent
    / "reports"
    / "models"
    / "fraud"
    / "tuning"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


fraud = pd.read_csv(
    PROCESSED_DIR
    / "fraud_modeling_base.csv"
)

In [4]:
# ============================================================
# CHARGER / RÉPARER LE FEATURE MANIFEST FRAUD
# ============================================================

import pandas as pd
import json


# ------------------------------------------------------------
# 1. S'assurer que le dataset Fraud est chargé
# ------------------------------------------------------------

if "fraud" not in globals():

    fraud = pd.read_csv(
        PROCESSED_DIR
        / "fraud_modeling_base.csv"
    )

    print(
        "Dataset Fraud rechargé :",
        fraud.shape
    )


# ------------------------------------------------------------
# 2. Charger le feature_manifest existant
# ------------------------------------------------------------

FEATURE_MANIFEST_PATH = (
    METADATA_DIR
    / "feature_manifest.json"
)


with open(
    FEATURE_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    feature_manifest = json.load(f)


print(
    "Clés actuelles du manifest :",
    list(feature_manifest.keys())
)


# ------------------------------------------------------------
# 3. Créer la configuration Fraud si elle est absente
# ------------------------------------------------------------

if "fraud" not in feature_manifest:

    print(
        "\n⚠️ Configuration Fraud absente."
    )

    print(
        "Création de la configuration Fraud..."
    )


    # ========================================================
    # VARIABLES CONTINUES
    # ========================================================

    continuous_features = [
        "Age",
        "Deductible"
    ]


    # ========================================================
    # VARIABLES ORDINALES / NUMÉRIQUES DISCRÈTES
    # ========================================================

    ordinal_features = [
        "WeekOfMonth",
        "WeekOfMonthClaimed",
        "DriverRating"
    ]


    # ========================================================
    # VARIABLES À EXCLURE
    # ========================================================

    excluded_features = {

        # Identifiant
        "PolicyNumber",

        # Target
        "FraudFound_P",

        # Utilisée pour le split temporel
        "Year",

        # Proxy / review variable
        "RepNumber",

        # Redondante
        "PolicyType",

        # Variables potentiellement indisponibles
        # au moment de la prédiction
        "Fault",
        "PoliceReportFiled",
        "WitnessPresent",
        "NumberOfSuppliments",
        "Days_Policy_Accident",
        "Days_Policy_Claim",
        "AddressChange_Claim"
    }


    # ========================================================
    # VÉRIFIER LES COLONNES IMPORTANTES
    # ========================================================

    required_columns = (
        continuous_features
        +
        ordinal_features
        +
        [
            "FraudFound_P",
            "Year"
        ]
    )


    missing_required = [
        col
        for col in required_columns
        if col not in fraud.columns
    ]


    if missing_required:

        raise ValueError(
            "Colonnes requises absentes du dataset Fraud : "
            f"{missing_required}"
        )


    # ========================================================
    # LES AUTRES FEATURES DEVIENNENT CATÉGORIELLES
    # ========================================================

    categorical_features = [
        col
        for col in fraud.columns
        if (
            col not in excluded_features
            and col not in continuous_features
            and col not in ordinal_features
        )
    ]


    # ========================================================
    # CONFIGURATION FRAUD
    # ========================================================

    fraud_config = {

        "target":
            "FraudFound_P",

        "split_column":
            "Year",

        "continuous_features":
            continuous_features,

        "ordinal_features":
            ordinal_features,

        "categorical_features":
            categorical_features,

        "excluded_features":
            sorted(
                excluded_features
            )
    }


    # ========================================================
    # AJOUT AU MANIFEST
    # ========================================================

    feature_manifest[
        "fraud"
    ] = fraud_config


    # ========================================================
    # SAUVEGARDE DÉFINITIVE
    # ========================================================

    with open(
        FEATURE_MANIFEST_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            feature_manifest,
            f,
            indent=4,
            ensure_ascii=False
        )


    print(
        "✅ Configuration Fraud ajoutée "
        "et sauvegardée dans feature_manifest.json"
    )


# ------------------------------------------------------------
# 4. Charger maintenant la configuration Fraud
# ------------------------------------------------------------

fraud_config = (
    feature_manifest[
        "fraud"
    ]
)


# ------------------------------------------------------------
# 5. Extraire les groupes de features
# ------------------------------------------------------------

continuous_features = (
    fraud_config[
        "continuous_features"
    ]
)


ordinal_features = (
    fraud_config[
        "ordinal_features"
    ]
)


categorical_features = (
    fraud_config[
        "categorical_features"
    ]
)


features = (
    continuous_features
    +
    ordinal_features
    +
    categorical_features
)


# ------------------------------------------------------------
# 6. Vérification anti-leakage
# ------------------------------------------------------------

forbidden_features = {
    "PolicyNumber",
    "FraudFound_P",
    "Year",
    "RepNumber",
    "PolicyType",
    "Fault",
    "PoliceReportFiled",
    "WitnessPresent",
    "NumberOfSuppliments",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "AddressChange_Claim"
}


unexpected_features = (
    forbidden_features
    .intersection(
        set(features)
    )
)


assert len(
    unexpected_features
) == 0, (
    "❌ Variables interdites présentes : "
    f"{unexpected_features}"
)


# ------------------------------------------------------------
# 7. Affichage final
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "FRAUD FEATURE CONFIGURATION"
)

print(
    "========================================"
)


print(
    "\nContinuous features :"
)

print(
    continuous_features
)


print(
    "\nOrdinal features :"
)

print(
    ordinal_features
)


print(
    "\nCategorical features :"
)

for col in categorical_features:

    print(
        "-",
        col
    )


print(
    "\nTotal features :",
    len(features)
)


print(
    "\nExcluded features :"
)

for col in fraud_config[
    "excluded_features"
]:

    print(
        "-",
        col
    )


print(
    "\nLeakage check : OK"
)


print(
    "\nClés finales du manifest :",
    list(
        feature_manifest.keys()
    )
)


print(
    "\n✅ Configuration Fraud prête."
)

Clés actuelles du manifest : ['claim_occurrence', 'claim_frequency', 'fraud_detection']

⚠️ Configuration Fraud absente.
Création de la configuration Fraud...
✅ Configuration Fraud ajoutée et sauvegardée dans feature_manifest.json

FRAUD FEATURE CONFIGURATION

Continuous features :
['Age', 'Deductible']

Ordinal features :
['WeekOfMonth', 'WeekOfMonthClaimed', 'DriverRating']

Categorical features :
- Month
- DayOfWeek
- Make
- AccidentArea
- DayOfWeekClaimed
- MonthClaimed
- Sex
- MaritalStatus
- VehicleCategory
- VehiclePrice
- PastNumberOfClaims
- AgeOfVehicle
- AgeOfPolicyHolder
- AgentType
- NumberOfCars
- BasePolicy

Total features : 21

Excluded features :
- AddressChange_Claim
- Days_Policy_Accident
- Days_Policy_Claim
- Fault
- FraudFound_P
- NumberOfSuppliments
- PoliceReportFiled
- PolicyNumber
- PolicyType
- RepNumber
- WitnessPresent
- Year

Leakage check : OK

Clés finales du manifest : ['claim_occurrence', 'claim_frequency', 'fraud_detection', 'fraud']

✅ Configuration F

In [5]:
with open(
    METADATA_DIR / "feature_manifest.json",
    "r",
    encoding="utf-8"
) as f:

    verification_manifest = json.load(f)


print(
    verification_manifest.keys()
)

assert "fraud" in verification_manifest

print(
    "✅ La clé fraud est réellement enregistrée sur disque."
)

dict_keys(['claim_occurrence', 'claim_frequency', 'fraud_detection', 'fraud'])
✅ La clé fraud est réellement enregistrée sur disque.


In [6]:
with open(
    METADATA_DIR / "feature_manifest.json",
    "r",
    encoding="utf-8"
) as f:
    feature_manifest = json.load(f)

fraud_config = feature_manifest["fraud"]

In [9]:
fraud_train = (
    fraud[
        fraud["Year"] == 1994
    ]
    .copy()
)

fraud_validation = (
    fraud[
        fraud["Year"] == 1995
    ]
    .copy()
)

fraud_test = (
    fraud[
        fraud["Year"] == 1996
    ]
    .copy()
)


X_train = fraud_train[
    features
].copy()

y_train = fraud_train[
    "FraudFound_P"
].astype(int)


X_validation = fraud_validation[
    features
].copy()

y_validation = fraud_validation[
    "FraudFound_P"
].astype(int)

In [10]:
print(
    "TRAIN :",
    X_train.shape,
    "| frauds =",
    y_train.sum()
)

print(
    "VALIDATION :",
    X_validation.shape,
    "| frauds =",
    y_validation.sum()
)

print(
    "TEST 1996 :",
    fraud_test.shape
)

TRAIN : (6142, 21) | frauds = 409
VALIDATION : (5195, 21) | frauds = 301
TEST 1996 : (4083, 25)


In [11]:
continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        )
    ]
)


ordinal_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_pipeline,
            continuous_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [12]:
inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [13]:
xgb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
    
)

In [14]:
xgb_params = {

    "model__n_estimators": [
        100,
        200,
        300,
        500
    ],

    "model__learning_rate": [
        0.02,
        0.05,
        0.08,
        0.10
    ],

    "model__max_depth": [
        2,
        3,
        4,
        5,
        6
    ],

    "model__min_child_weight": [
        1,
        3,
        5,
        10
    ],

    "model__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "model__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "model__reg_alpha": [
        0.0,
        0.01,
        0.1,
        1.0
    ],

    "model__reg_lambda": [
        1.0,
        2.0,
        5.0,
        10.0
    ],

    "model__scale_pos_weight": [
        1.0,
        2.0,
        4.0,
        7.0,
        10.0,
        14.0
    ]
}

In [15]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_params,
    n_iter=30,
    scoring="average_precision",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=2,
    return_train_score=True
)

In [16]:
start = time.perf_counter()

xgb_search.fit(
    X_train,
    y_train
)

xgb_tuning_time = (
    time.perf_counter()
    -
    start
)


print(
    "Best XGB params :",
    xgb_search.best_params_
)

print(
    "Best XGB CV PR-AUC :",
    xgb_search.best_score_
)

print(
    "Time :",
    xgb_tuning_time
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END model__colsample_bytree=0.7, model__learning_rate=0.1, model__max_depth=6, model__min_child_weight=10, model__n_estimators=200, model__reg_alpha=0.1, model__reg_lambda=2.0, model__scale_pos_weight=7.0, model__subsample=0.9; total time=   1.4s
[CV] END model__colsample_bytree=0.7, model__learning_rate=0.1, model__max_depth=6, model__min_child_weight=10, model__n_estimators=200, model__reg_alpha=0.1, model__reg_lambda=2.0, model__scale_pos_weight=7.0, model__subsample=0.9; total time=   0.6s
[CV] END model__colsample_bytree=0.7, model__learning_rate=0.1, model__max_depth=6, model__min_child_weight=10, model__n_estimators=200, model__reg_alpha=0.1, model__reg_lambda=2.0, model__scale_pos_weight=7.0, model__subsample=0.9; total time=   0.5s
[CV] END model__colsample_bytree=0.7, model__learning_rate=0.1, model__max_depth=6, model__min_child_weight=10, model__n_estimators=200, model__reg_alpha=0.1, model__reg_lambda=2.0, 

In [17]:
rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [18]:
rf_params = {

    "model__n_estimators": [
        200,
        400,
        600,
        800
    ],

    "model__max_depth": [
        None,
        5,
        8,
        12,
        16
    ],

    "model__min_samples_split": [
        2,
        5,
        10,
        20
    ],

    "model__min_samples_leaf": [
        1,
        2,
        5,
        10,
        20
    ],

    "model__max_features": [
        "sqrt",
        "log2",
        0.5
    ],

    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}

In [19]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter=25,
    scoring="average_precision",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=2,
    return_train_score=True
)

In [20]:
start = time.perf_counter()

rf_search.fit(
    X_train,
    y_train
)

rf_tuning_time = (
    time.perf_counter()
    -
    start
)


print(
    "Best RF params :",
    rf_search.best_params_
)

print(
    "Best RF CV PR-AUC :",
    rf_search.best_score_
)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV] END model__class_weight=balanced_subsample, model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=10, model__min_samples_split=5, model__n_estimators=600; total time=   4.0s
[CV] END model__class_weight=balanced_subsample, model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=10, model__min_samples_split=5, model__n_estimators=600; total time=   3.7s
[CV] END model__class_weight=balanced_subsample, model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=10, model__min_samples_split=5, model__n_estimators=600; total time=   4.2s
[CV] END model__class_weight=balanced_subsample, model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=10, model__min_samples_split=5, model__n_estimators=600; total time=   3.6s
[CV] END model__class_weight=balanced_subsample, model__max_depth=12, model__max_features=sqrt, model__min_samples_leaf=10, model__min_samples_split=5, mo

In [21]:
pd.DataFrame(
    xgb_search.cv_results_
).to_csv(
    REPORT_DIR
    / "xgboost_tuning_results.csv",
    index=False
)


pd.DataFrame(
    rf_search.cv_results_
).to_csv(
    REPORT_DIR
    / "random_forest_tuning_results.csv",
    index=False
)

In [22]:
best_xgb = (
    xgb_search.best_estimator_
)

best_rf = (
    rf_search.best_estimator_
)

In [23]:
xgb_raw_proba = (
    best_xgb
    .predict_proba(
        X_validation
    )[:, 1]
)


rf_raw_proba = (
    best_rf
    .predict_proba(
        X_validation
    )[:, 1]
)

In [24]:
def fraud_probability_metrics(
    name,
    y_true,
    probabilities
):

    return {
        "model":
            name,

        "pr_auc":
            average_precision_score(
                y_true,
                probabilities
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probabilities
            ),

        "brier":
            brier_score_loss(
                y_true,
                probabilities
            ),

        "log_loss":
            log_loss(
                y_true,
                probabilities
            )
    }

In [25]:
xgb_sigmoid = CalibratedClassifierCV(
    estimator=xgb_search.best_estimator_,
    method="sigmoid",
    cv=5
)

xgb_sigmoid.fit(
    X_train,
    y_train
)

xgb_sigmoid_proba = (
    xgb_sigmoid
    .predict_proba(
        X_validation
    )[:, 1]
)

In [26]:
xgb_isotonic = CalibratedClassifierCV(
    estimator=xgb_search.best_estimator_,
    method="isotonic",
    cv=5
)

xgb_isotonic.fit(
    X_train,
    y_train
)

xgb_isotonic_proba = (
    xgb_isotonic
    .predict_proba(
        X_validation
    )[:, 1]
)

In [27]:
rf_sigmoid = CalibratedClassifierCV(
    estimator=rf_search.best_estimator_,
    method="sigmoid",
    cv=5
)

rf_sigmoid.fit(
    X_train,
    y_train
)

rf_sigmoid_proba = (
    rf_sigmoid
    .predict_proba(
        X_validation
    )[:, 1]
)

In [28]:
rf_isotonic = CalibratedClassifierCV(
    estimator=rf_search.best_estimator_,
    method="isotonic",
    cv=5
)

rf_isotonic.fit(
    X_train,
    y_train
)

rf_isotonic_proba = (
    rf_isotonic
    .predict_proba(
        X_validation
    )[:, 1]
)

In [29]:
calibration_comparison = pd.DataFrame([
    fraud_probability_metrics(
        "xgb_raw",
        y_validation,
        xgb_raw_proba
    ),

    fraud_probability_metrics(
        "xgb_sigmoid",
        y_validation,
        xgb_sigmoid_proba
    ),

    fraud_probability_metrics(
        "xgb_isotonic",
        y_validation,
        xgb_isotonic_proba
    ),

    fraud_probability_metrics(
        "rf_raw",
        y_validation,
        rf_raw_proba
    ),

    fraud_probability_metrics(
        "rf_sigmoid",
        y_validation,
        rf_sigmoid_proba
    ),

    fraud_probability_metrics(
        "rf_isotonic",
        y_validation,
        rf_isotonic_proba
    )
])

In [30]:
calibration_comparison.sort_values(
    "pr_auc",
    ascending=False
).round(6)

,model,pr_auc,roc_auc,brier,log_loss
5,rf_isotonic,0.090384,0.671002,0.053816,0.210475
0,xgb_raw,0.085368,0.607108,0.082520,0.296569
2,xgb_isotonic,0.081573,0.614711,0.054295,0.218508
4,rf_sigmoid,0.081003,0.658716,0.054346,0.219009
3,rf_raw,0.078185,0.648283,0.054401,0.213429
1,xgb_sigmoid,0.044908,0.411245,0.055316,0.227012


In [31]:
calibration_comparison.to_csv(
    REPORT_DIR
    / "fraud_calibration_comparison.csv",
    index=False
)

In [32]:
print(
    "=" * 80
)

print(
    "FRAUD — TUNING & CALIBRATION RESULTS"
)

print(
    "=" * 80
)


print(
    "\n=== DATA ==="
)

print(
    "1994 training observations :",
    len(y_train)
)

print(
    "1994 frauds :",
    y_train.sum()
)

print(
    "1995 validation observations :",
    len(y_validation)
)

print(
    "1995 frauds :",
    y_validation.sum()
)


print(
    "\n=== XGBOOST TUNING ==="
)

print(
    "Best CV PR-AUC :",
    xgb_search.best_score_
)

for key, value in (
    xgb_search.best_params_.items()
):
    print(
        key,
        "=",
        value
    )


print(
    "\n=== RANDOM FOREST TUNING ==="
)

print(
    "Best CV PR-AUC :",
    rf_search.best_score_
)

for key, value in (
    rf_search.best_params_.items()
):
    print(
        key,
        "=",
        value
    )


print(
    "\n=== 1995 CALIBRATION COMPARISON ==="
)

print(
    calibration_comparison
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n=== VALIDATION PREVALENCE ==="
)

print(
    y_validation.mean()
)


print(
    "\n=== PR-AUC LIFTS ==="
)

for _, row in (
    calibration_comparison.iterrows()
):

    print(
        row["model"],
        "->",
        round(
            row["pr_auc"]
            /
            y_validation.mean(),
            3
        ),
        "x"
    )


print(
    "\n1996 TEST UTILISÉ ? NON"
)

print(
    "=" * 80
)

FRAUD — TUNING & CALIBRATION RESULTS

=== DATA ===
1994 training observations : 6142
1994 frauds : 409
1995 validation observations : 5195
1995 frauds : 301

=== XGBOOST TUNING ===
Best CV PR-AUC : 0.21463937522236537
model__subsample = 1.0
model__scale_pos_weight = 4.0
model__reg_lambda = 5.0
model__reg_alpha = 0.0
model__n_estimators = 200
model__min_child_weight = 10
model__max_depth = 3
model__learning_rate = 0.1
model__colsample_bytree = 0.7

=== RANDOM FOREST TUNING ===
Best CV PR-AUC : 0.21213632626666007
model__n_estimators = 800
model__min_samples_split = 5
model__min_samples_leaf = 10
model__max_features = log2
model__max_depth = 16
model__class_weight = None

=== 1995 CALIBRATION COMPARISON ===
       model   pr_auc  roc_auc    brier  log_loss
 rf_isotonic 0.090384 0.671002 0.053816  0.210475
     xgb_raw 0.085368 0.607108 0.082520  0.296569
xgb_isotonic 0.081573 0.614711 0.054295  0.218508
  rf_sigmoid 0.081003 0.658716 0.054346  0.219009
      rf_raw 0.078185 0.648283 0.05

# Analyse du tuning et de la calibration — Fraud Classification

## Tuning interne sur 1994

Le tuning réalisé uniquement sur les données de 1994 obtient :

### XGBoost

PR-AUC moyenne en validation croisée :

**0.21464**

Meilleure configuration :

- n_estimators = 200
- learning_rate = 0.10
- max_depth = 3
- min_child_weight = 10
- subsample = 1.0
- colsample_bytree = 0.7
- reg_alpha = 0
- reg_lambda = 5
- scale_pos_weight = 4

### Random Forest

PR-AUC moyenne en validation croisée :

**0.21214**

Meilleure configuration :

- n_estimators = 800
- max_depth = 16
- min_samples_split = 5
- min_samples_leaf = 10
- max_features = log2
- class_weight = None

Les deux modèles présentent donc des performances proches
durant la validation croisée interne à l'année 1994.

---

# Validation temporelle sur 1995

Les modèles ont ensuite été évalués sur l'année 1995,
qui n'a pas participé au tuning des hyperparamètres.

| Configuration | PR-AUC | ROC-AUC | Brier | Log Loss |
|---|---:|---:|---:|---:|
| Random Forest + Isotonic | **0.09038** | **0.67100** | **0.05382** | **0.21048** |
| XGBoost Raw | 0.08537 | 0.60711 | 0.08252 | 0.29657 |
| XGBoost + Isotonic | 0.08157 | 0.61471 | 0.05430 | 0.21851 |
| Random Forest + Sigmoid | 0.08100 | 0.65872 | 0.05435 | 0.21901 |
| Random Forest Raw | 0.07819 | 0.64828 | 0.05440 | 0.21343 |
| XGBoost + Sigmoid | 0.04491 | 0.41125 | 0.05532 | 0.22701 |

La prévalence de fraude en 1995 est :

**0.05794**

Random Forest calibré avec isotonic obtient donc une PR-AUC
environ :

**1.56 fois supérieure à la prévalence.**

---

## Écart entre Cross-Validation et validation temporelle

Les PR-AUC obtenues durant le tuning sur 1994 dépassent 0.21,
alors que les performances sur 1995 se situent autour de 0.08–0.09.

Cet écart montre que la généralisation temporelle de la détection
de fraude est beaucoup plus difficile que ne le laisse supposer
une validation interne réalisée dans une seule année.

Plusieurs facteurs peuvent contribuer à cette différence :

- nombre limité de fraudes disponibles ;
- classe fortement déséquilibrée ;
- évolution temporelle du taux de fraude ;
- changement possible des relations entre les variables et la cible.

Cette observation justifie l'utilisation d'un protocole temporel.

---

## Calibration

XGBoost brut possède une PR-AUC intéressante mais un Brier Score
élevé :

**0.08252**

Après calibration isotonic, le Brier Score devient :

**0.05430**

ce qui constitue une amélioration importante de la qualité
probabiliste.

Random Forest présente déjà un Brier relativement faible à l'état brut,
mais la calibration isotonic améliore également ses performances.

La configuration Random Forest + Isotonic obtient simultanément :

- la meilleure PR-AUC ;
- le meilleur ROC-AUC ;
- le meilleur Brier Score ;
- un Log Loss compétitif.

---

# Décision provisoire

Le candidat principal retenu avant l'étude du seuil est :

## Random Forest + Isotonic Calibration

XGBoost est conservé comme modèle secondaire de comparaison.

La configuration XGBoost + Sigmoid est écartée en raison de sa mauvaise
généralisation temporelle sur 1995.

Le test final 1996 n'a toujours pas été utilisé.

In [33]:
selected_model_name = "rf_isotonic"

selected_validation_proba = (
    rf_isotonic_proba
)

print(
    "Selected model :",
    selected_model_name
)

print(
    "Validation probability range :",
    selected_validation_proba.min(),
    "->",
    selected_validation_proba.max()
)

Selected model : rf_isotonic
Validation probability range : 0.0025 -> 0.3450364916359879


In [34]:
from sklearn.metrics import precision_recall_curve


precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_validation,
        selected_validation_proba
    )
)


threshold_table = pd.DataFrame({
    "threshold":
        thresholds,

    "precision":
        precision_values[:-1],

    "recall":
        recall_values[:-1]
})

In [35]:
threshold_table[
    "f1"
] = (
    2
    *
    threshold_table[
        "precision"
    ]
    *
    threshold_table[
        "recall"
    ]
    /
    (
        threshold_table[
            "precision"
        ]
        +
        threshold_table[
            "recall"
        ]
        +
        1e-12
    )
)

In [36]:
best_f1_index = (
    threshold_table[
        "f1"
    ]
    .idxmax()
)


best_f1_row = (
    threshold_table
    .loc[
        best_f1_index
    ]
)


TECHNICAL_THRESHOLD = float(
    best_f1_row[
        "threshold"
    ]
)


print(
    "Best F1 threshold :",
    TECHNICAL_THRESHOLD
)

print(
    best_f1_row
)

Best F1 threshold : 0.08585764735167348
threshold    0.085858
precision    0.096984
recall       0.694352
f1           0.170195
Name: 548, dtype: float64


In [38]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

print("✅ Métriques sklearn importées")

✅ Métriques sklearn importées


In [39]:
validation_pred = (
    selected_validation_proba
    >= TECHNICAL_THRESHOLD
).astype(int)


validation_cm = confusion_matrix(
    y_validation,
    validation_pred
)


tn, fp, fn, tp = validation_cm.ravel()


print(validation_cm)

print(
    "TN :",
    tn
)

print(
    "FP :",
    fp
)

print(
    "FN :",
    fn
)

print(
    "TP :",
    tp
)

[[2948 1946]
 [  92  209]]
TN : 2948
FP : 1946
FN : 92
TP : 209


In [40]:
validation_precision = precision_score(
    y_validation,
    validation_pred,
    zero_division=0
)

validation_recall = recall_score(
    y_validation,
    validation_pred,
    zero_division=0
)

validation_f1 = f1_score(
    y_validation,
    validation_pred,
    zero_division=0
)


print(
    "Precision :",
    validation_precision
)

print(
    "Recall :",
    validation_recall
)

print(
    "F1 :",
    validation_f1
)

Precision : 0.09698375870069606
Recall : 0.6943521594684385
F1 : 0.17019543973941367


In [41]:
recall_targets = [
    0.30,
    0.50,
    0.70,
    0.80
]


recall_scenarios = []


for target_recall in recall_targets:

    candidates = (
        threshold_table[
            threshold_table[
                "recall"
            ]
            >= target_recall
        ]
        .copy()
    )


    if not candidates.empty:

        best_candidate = (
            candidates
            .sort_values(
                "precision",
                ascending=False
            )
            .iloc[0]
        )


        recall_scenarios.append({
            "target_recall":
                target_recall,

            "threshold":
                best_candidate[
                    "threshold"
                ],

            "precision":
                best_candidate[
                    "precision"
                ],

            "recall":
                best_candidate[
                    "recall"
                ],

            "f1":
                best_candidate[
                    "f1"
                ]
        })

In [42]:
recall_scenarios_df = pd.DataFrame(
    recall_scenarios
)

recall_scenarios_df

,target_recall,threshold,precision,recall,f1
0,0.3,0.085858,0.096984,0.694352,0.170195
1,0.5,0.085858,0.096984,0.694352,0.170195
2,0.7,0.082007,0.088332,0.807309,0.159240
3,0.8,0.082007,0.088332,0.807309,0.159240


In [43]:
threshold_table.to_csv(
    REPORT_DIR
    / "fraud_threshold_analysis.csv",
    index=False
)


recall_scenarios_df.to_csv(
    REPORT_DIR
    / "fraud_recall_scenarios.csv",
    index=False
)

In [44]:
best_rf_params = (
    rf_search.best_params_
)

In [45]:
rf_parameters = {
    key.replace(
        "model__",
        ""
    ):
    value

    for key, value
    in best_rf_params.items()
}

In [46]:
fraud_candidate = {

    "task":
        "fraud_classification",

    "model":
        "random_forest",

    "best_parameters":
        rf_parameters,

    "calibration":
        "isotonic",

    "validation_pr_auc":
        float(
            average_precision_score(
                y_validation,
                selected_validation_proba
            )
        ),

    "validation_roc_auc":
        float(
            roc_auc_score(
                y_validation,
                selected_validation_proba
            )
        ),

    "validation_brier":
        float(
            brier_score_loss(
                y_validation,
                selected_validation_proba
            )
        ),

    "technical_threshold":
        TECHNICAL_THRESHOLD,

    "threshold_policy":
        "maximum_f1_on_1995_validation",

    "training_year":
        1994,

    "validation_year":
        1995,

    "test_year":
        1996,

    "test_used":
        False
}

In [47]:
with open(
    METADATA_DIR
    / "fraud_candidate.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        fraud_candidate,
        f,
        indent=4,
        ensure_ascii=False
    )

In [48]:
print(
    "=" * 80
)

print(
    "FRAUD — MODEL SELECTION & THRESHOLD RESULTS"
)

print(
    "=" * 80
)


print(
    "\n=== SELECTED MODEL ==="
)

print(
    "Random Forest + Isotonic"
)


print(
    "\n=== VALIDATION PROBABILITY METRICS ==="
)

print(
    "PR-AUC :",
    average_precision_score(
        y_validation,
        selected_validation_proba
    )
)

print(
    "ROC-AUC :",
    roc_auc_score(
        y_validation,
        selected_validation_proba
    )
)

print(
    "Brier :",
    brier_score_loss(
        y_validation,
        selected_validation_proba
    )
)

print(
    "Log Loss :",
    log_loss(
        y_validation,
        selected_validation_proba
    )
)


print(
    "\n=== TECHNICAL THRESHOLD ==="
)

print(
    "Threshold :",
    TECHNICAL_THRESHOLD
)

print(
    "Precision :",
    validation_precision
)

print(
    "Recall :",
    validation_recall
)

print(
    "F1 :",
    validation_f1
)


print(
    "\n=== CONFUSION MATRIX ==="
)

print(
    validation_cm
)

print(
    "TN =",
    tn
)

print(
    "FP =",
    fp
)

print(
    "FN =",
    fn
)

print(
    "TP =",
    tp
)


print(
    "\n=== RECALL SCENARIOS ==="
)

print(
    recall_scenarios_df
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n=== RANDOM FOREST PARAMETERS ==="
)

for key, value in (
    rf_parameters.items()
):

    print(
        key,
        "=",
        value
    )


print(
    "\n1996 TEST UTILISÉ ? NON"
)

print(
    "MODEL CONFIGURATION LOCKED BEFORE TEST : OUI"
)

print(
    "=" * 80
)

FRAUD — MODEL SELECTION & THRESHOLD RESULTS

=== SELECTED MODEL ===
Random Forest + Isotonic

=== VALIDATION PROBABILITY METRICS ===
PR-AUC : 0.09038410368422278
ROC-AUC : 0.6710019862955114
Brier : 0.05381599102173139
Log Loss : 0.2104749309760136

=== TECHNICAL THRESHOLD ===
Threshold : 0.08585764735167348
Precision : 0.09698375870069606
Recall : 0.6943521594684385
F1 : 0.17019543973941367

=== CONFUSION MATRIX ===
[[2948 1946]
 [  92  209]]
TN = 2948
FP = 1946
FN = 92
TP = 209

=== RECALL SCENARIOS ===
 target_recall  threshold  precision   recall       f1
           0.3   0.085858   0.096984 0.694352 0.170195
           0.5   0.085858   0.096984 0.694352 0.170195
           0.7   0.082007   0.088332 0.807309 0.159240
           0.8   0.082007   0.088332 0.807309 0.159240

=== RANDOM FOREST PARAMETERS ===
n_estimators = 800
min_samples_split = 5
min_samples_leaf = 10
max_features = log2
max_depth = 16
class_weight = None

1996 TEST UTILISÉ ? NON
MODEL CONFIGURATION LOCKED BEFORE TES

# Sélection finale du candidat Fraud

Après tuning et comparaison des méthodes de calibration sur l'année 1995,
le modèle retenu est :

**Random Forest + calibration isotonic**

La configuration Random Forest est :

- n_estimators = 800
- max_depth = 16
- min_samples_split = 5
- min_samples_leaf = 10
- max_features = log2
- class_weight = None

## Performances probabilistes sur 1995

- PR-AUC = **0.09038**
- ROC-AUC = **0.67100**
- Brier Score = **0.05382**
- Log Loss = **0.21047**

La prévalence de fraude en 1995 est de 5,79 %.

La PR-AUC est donc environ 1,56 fois supérieure à la prévalence
de la classe positive.

Le modèle possède ainsi un pouvoir de classement réel,
mais les performances restent modestes.

Le système doit être considéré comme un outil de priorisation
des dossiers suspects plutôt que comme un système capable
de déterminer automatiquement qu'une déclaration est frauduleuse.

## Seuil technique

Le seuil maximisant le F1-score sur 1995 est :

**0.08586**

À ce seuil :

- Precision = **9.70 %**
- Recall = **69.44 %**
- F1 = **17.02 %**

La matrice de confusion est :

| | Prédit non fraude | Prédit fraude |
|---|---:|---:|
| Réel non fraude | 2948 | 1946 |
| Réel fraude | 92 | 209 |

Le modèle détecte donc 209 fraudes sur 301 mais génère
1 946 faux positifs.

Environ 41,5 % des dossiers seraient signalés pour investigation
avec ce seuil.

Ce coût opérationnel potentiel montre que le seuil maximisant F1
ne doit pas être interprété comme un seuil métier optimal.

## Scénarios de Recall

Pour obtenir un Recall d'environ 80 %, le seuil doit être abaissé
à environ 0.082.

Dans ce cas :

- Recall ≈ 80.73 %
- Precision ≈ 8.83 %
- F1 ≈ 15.92 %

L'augmentation du Recall entraîne donc davantage de faux positifs.

Le choix métier final devrait idéalement intégrer le coût d'une fraude
non détectée et le coût d'une investigation inutile.

Ces coûts ne sont pas disponibles dans le dataset.

## Configuration verrouillée

La configuration utilisée pour l'évaluation finale est donc :

- modèle : Random Forest ;
- calibration : isotonic ;
- seuil technique : 0.085858 ;
- règle du seuil : maximum du F1 sur 1995.

L'année 1996 n'a pas été utilisée pour le tuning,
la sélection du modèle, la calibration ou le choix du seuil.

Sa prévalence globale avait cependant été inspectée précédemment
à titre descriptif.